In [8]:
import pandas as pd
import numpy as np
import random
import warnings

# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)


# Backtest trades function
def backtest_trades(price_data, signal_data, tp=None, sl=None, entry_time_offset=None,
                    percentage_change=None, time_limit_minutes=None, ignore_time_interval_before=None, ignore_time_interval_after=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'Signal Open Price', 'Entry Price', 'TP Price', 'SL Price', 'Result', 'Duration',
        'Execution Latency', 'ROI', 'NAV', 'Ignore Reason'
    ])
    
    initial_margin = 100000
    current_margin = initial_margin
    exit_datetimes = []
    #ignore_intervals = []
    # Initialize flags to track if 1 or -1 has already been seen
    seen_signal = False
    
    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']
        # btc_value = row['BTC']
        # 
        # if btc_value == 'up':
        #     ignore_intervals.append((signal_datetime, signal_datetime + pd.Timedelta(days=2), 'Sell'))
        # elif btc_value == 'down':
        #     ignore_intervals.append((signal_datetime, signal_datetime + pd.Timedelta(days=2), 'Buy'))

          
        if signal_value == 0:
            seen_signal = False
            continue
        
        if not seen_signal:
            if signal_value == 1:
                side = 'Buy'
                seen_signal = True
            elif signal_value == -1:
                side = 'Sell'
                seen_signal = True
            else:
                continue
            
        
        adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)

        signal_open_price = price_data.at[adjusted_signal_datetime, 'Open']
        
        
        # ignore_signal = False
        # reason = ''
        # for start, end, ignore_side in ignore_intervals:
        #     if start <= signal_datetime <= end and side == ignore_side:
        #         ignore_signal = True
        #         reason = f'Ignored due to BTC {ignore_side.lower()} signal from {start} to {end}'
        #         break
        # 
        # if ignore_signal:
        #     new_row = pd.DataFrame([{
        #         'Datetime': signal_datetime,
        #         'Side': side,
        #         'Signal Open Price': signal_open_price,
        #         'Entry Price': None,
        #         'TP Price': None,
        #         'SL Price': None,
        #         'Result': 'Ignored',
        #         'Duration': '00:00:00',
        #         'Execution Latency': '00:00:00',
        #         'ROI': 0,
        #         'NAV': current_margin,
        #         'Ignore Reason': reason
        #     }])
        #     output_data = pd.concat([output_data, new_row], ignore_index=True)
        #     continue
        
        # Ignoring signals based on open trades
        exit_datetimes.sort(key=lambda x: x[0])
        ignore_signal = False
        reason = ''
        if exit_datetimes:
            later_exits = [ed for ed in exit_datetimes if ed[0] > signal_datetime]


            if len(later_exits) >= 3:
                result = 'Ignored'
                reason = 'More than 2 open trades'
                ignore_signal = True
            elif len(later_exits) == 2:
                if later_exits[-1][1] == side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'Two open trades, last one with different side'
                    ignore_signal = True
            elif len(later_exits) == 1:
                if later_exits[-1][1] != side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'One open trade with the same side'
                    ignore_signal = True


        if ignore_signal:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': result,
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': reason
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        #logic: ignore signals within a specific time interval before and after events
        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            ignore_signal = any(event_datetime - pd.Timedelta(minutes=ignore_time_interval_before) <= signal_datetime <= event_datetime + pd.Timedelta(minutes=ignore_time_interval_after)
                                for event_datetime in event_times)
            if ignore_signal:
                new_row = pd.DataFrame([{
                    'Datetime': signal_datetime,
                    'Side': side,
                    'Signal Open Price': signal_open_price,
                    'Entry Price': None,
                    'TP Price': None,
                    'SL Price': None,
                    'Result': 'Ignored',
                    'Duration': '00:00:00',
                    'Execution Latency': '00:00:00',
                    'ROI': 0,
                    'NAV': current_margin,
                    'Ignore Reason': 'Signal around economic event'
                }])
                output_data = pd.concat([output_data, new_row], ignore_index=True)
                continue

        entry_datetime, entry_price, entry_duration = determine_entry(price_data, signal_datetime, percentage_change,
                                                                      side, time_limit_minutes, entry_time_offset)
        if entry_datetime is None:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': 'Not Filled',
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': ''
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue
        
        # Update NAV on filled order
        current_margin *= (1 - 0.0002)
        
        if side == 'Buy':
            tp_price = entry_price * (1 + tp)
            sl_price = entry_price * (1 - sl)
        else:
            tp_price = entry_price * (1 - tp)
            sl_price = entry_price * (1 + sl)
        
        result, duration_str = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)
        exit_datetime = entry_datetime + pd.Timedelta(duration_str)

        if result in [1, -1]:
            exit_datetimes.append((exit_datetime, side))

        # Check for economic data event before the trade exit
        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            for event_datetime in event_times:
                if entry_datetime < event_datetime < exit_datetime:
                    exit_datetime = event_datetime - pd.Timedelta(minutes=10)
                    if exit_datetime in price_data.index:
                        exit_price = price_data.at[exit_datetime, 'Open']
                        result = 'ended before data'
                        if (side == 'Buy' and exit_price > entry_price) or (side == 'Sell' and exit_price < entry_price):
                            result += ' with profit'
                            pct_change = (exit_price - entry_price) / entry_price if side == 'Buy' else (entry_price - exit_price) / entry_price
                            current_margin = current_margin * (1 + pct_change)
                        else:
                            result += ' with loss'
                            pct_change = (entry_price - exit_price) / entry_price if side == 'Buy' else (exit_price - entry_price) / entry_price
                            current_margin = current_margin * (1 - pct_change)
                    break

        if result not in ['ended before data with profit', 'ended before data with loss', 'ended before data with no exact price']:
            if result == 1:
                current_margin = current_margin * (1 + tp)
                current_margin *= (1 - 0.0005)
            elif result == -1:
                current_margin = current_margin * (1 - sl)
                current_margin *= (1 - 0.0005)

        
        roi = ((current_margin - initial_margin) / initial_margin) * 100
        nav = current_margin
        initial_margin = current_margin
        
        new_row = pd.DataFrame([{
            'Datetime': signal_datetime,
            'Side': side,
            'Signal Open Price': signal_open_price,
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Result': result,
            'Duration': duration_str,
            'Execution Latency': format_duration(entry_duration),
            'ROI': roi,
            'NAV': nav,
            'Ignore Reason': ''
        }])
        
        output_data = pd.concat([output_data, new_row], ignore_index=True)
     
    # Ensure 'Datetime' column is in datetime format
    output_data['Datetime'] = pd.to_datetime(output_data['Datetime'])
    
        
        # Calculate Daily Return
    output_data['Date'] = output_data['Datetime'].dt.date
    daily_nav = output_data.groupby('Date')['NAV'].last().to_dict()
    daily_returns = {}
    previous_day_nav = 100000
    
    for date, nav in daily_nav.items():
        daily_return = (nav - previous_day_nav) / previous_day_nav * 100
        daily_returns[date] = daily_return
        previous_day_nav = nav

    output_data['Daily Return'] = output_data['Date'].map(daily_returns)
    output_data.drop(columns=['Date'], inplace=True)    
    
    return output_data


# Helper functions
def format_duration(duration):
    seconds = duration.total_seconds()
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"


def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None
    subsequent_prices = price_data.loc[entry_datetime:]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['Low'] <= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
        else:
            if price_row['Low'] <= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['High'] >= sl_price:
                result = -1
                exit_datetime = current_datetime
                break

    if exit_datetime:
        duration = exit_datetime - entry_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'

    return result, duration_str


def determine_entry(price_data, signal_datetime, percentage_change, side, time_limit_minutes, entry_time_offset):
    adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
    if adjusted_signal_datetime not in price_data.index:
        return None, None, None

    adjusted_open_price = price_data.at[adjusted_signal_datetime, 'Open']
    percentage_change_price = adjusted_open_price * (
                1 - percentage_change) if side == 'Buy' else adjusted_open_price * (1 + percentage_change)

    time_limit = adjusted_signal_datetime + pd.Timedelta(minutes=time_limit_minutes)
    subsequent_prices = price_data.loc[adjusted_signal_datetime:time_limit]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration

    return None, None, None

In [9]:
price_data = pd.read_csv('E:\SignalModel\price_2023-06-01_to_2024-07-31_min.csv', parse_dates=['Datetime'],
                         index_col='Datetime')
# Re-load the signal data without setting the index
signal_data = pd.read_csv('E:\Signal Backtesting\Input\sigai_signals_h.csv',
                          parse_dates=['Datetime'])

start_date = pd.Timestamp('2024-02-01')
end_date = pd.Timestamp('2024-05-01')
filtered_signal_data = signal_data[(signal_data['Datetime'] >= start_date) & (signal_data['Datetime'] <= end_date)]


In [11]:
# Define the parameters
tp = 0.012
sl = 0.012
entry_time_offset = 70 # Time offset in minutes
percentage_change = 0.0006
time_limit_minutes = 120 
# Call the backtest_trades function
backtest_output = backtest_trades(
    price_data=price_data,
    signal_data=filtered_signal_data,
    tp=tp,
    sl=sl,
    entry_time_offset=entry_time_offset,
    percentage_change=percentage_change,
    time_limit_minutes=time_limit_minutes,
    ignore_time_interval_before=720,
    ignore_time_interval_after=0
)
 
backtest_output

,Datetime,Side,Signal Open Price,Entry Price,TP Price,SL Price,Result,Duration,Execution Latency,ROI,NAV,Ignore Reason,Daily Return
0,2024-02-01 04:00:00,Buy,42070.7,42045.45758,42550.003071,41540.912089,1,09:40:00,00:00:00,1.12917,101129.170120,,1.129170
1,2024-02-01 05:00:00,Buy,42137.6,NaN,NaN,NaN,Ignored,00:00:00,00:00:00,0.00000,101129.170120,One open trade with the same side,1.129170
2,2024-02-01 06:00:00,Buy,42148.1,NaN,NaN,NaN,Ignored,00:00:00,00:00:00,0.00000,101129.170120,One open trade with the same side,1.129170
3,2024-02-01 07:00:00,Buy,42146.0,NaN,NaN,NaN,Ignored,00:00:00,00:00:00,0.00000,101129.170120,One open trade with the same side,1.129170
4,2024-02-01 08:00:00,Buy,42145.8,NaN,NaN,NaN,Ignored,00:00:00,00:00:00,0.00000,101129.170120,One open trade with the same side,1.129170
...,...,...,...,...,...,...,...,...,...,...,...,...,...
486,2024-04-30 19:00:00,Sell,60000.0,NaN,NaN,NaN,Ignored,00:00:00,00:00:00,0.00000,90031.333061,One open trade with the same side,0.973117
487,2024-04-30 20:00:00,Sell,60127.6,60163.67656,59441.712441,60885.640679,-1,01:17:00,00:07:00,-1.26915,88888.700289,,0.973117
488,2024-04-30 21:00:00,Sell,60311.8,NaN,NaN,NaN,Ignored,00:00:00,00:00:00,0.00000,88888.700289,One open trade with the same side,0.973117
489,2024-04-30 22:00:00,Sell,60393.6,NaN,NaN,NaN,Ignored,00:00:00,00:00:00,0.00000,88888.700289,One open trade with the same side,0.973117
